# AI-Powered Financial Fraud Detection & Risk Analytics System

## Notebook 5: Risk Scoring & Fraud Intelligence Engine

### Objective

The objective of this notebook is to generate fraud probabilities, calculate transaction risk scores, assign risk categories, create fraud investigation recommendations, and prepare the final dataset for reporting, dashboards, and business intelligence applications.

### Key Tasks

- Load trained fraud detection model
- Generate fraud probabilities
- Calculate risk scores
- Create risk categories
- Generate alert priorities
- Create investigation recommendations
- Generate fraud intelligence attributes
- Export risk-scored dataset

### Input

../data/processed/fraud_processed.csv

../models/random_forest_fraud_detector.pkl

### Output

../data/processed/fraud_risk_scored.csv

In [1]:
# Import Libraries

import pandas as pd
import numpy as np
import joblib

from sklearn.preprocessing import LabelEncoder

import warnings
warnings.filterwarnings("ignore")

In [2]:
# Load Dataset

df = pd.read_csv(
    "../data/processed/fraud_processed.csv"
)

print(f"Dataset Shape: {df.shape}")

df.head()

Dataset Shape: (6362620, 24)


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,...,balance_error_org,balance_error_dest,customer_type,merchant_type,age_group,customer_segment,channel,device_type,risk_region,account_tenure_months
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,...,1.455192e-11,-9839.64,C,M,26-35,Retail,Mobile App,iPhone,Central,20
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,...,-1.136868e-12,-1864.28,C,M,60+,Retail,Internet Banking,Android,East,7
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,...,0.000000e+00,-181.00,C,C,36-45,Retail,Internet Banking,Desktop,East,39
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,...,0.000000e+00,-21363.00,C,C,36-45,Premium,Mobile App,Android,North,116
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,...,0.000000e+00,-11668.14,C,M,18-25,Retail,Internet Banking,iPhone,East,19


In [3]:
# Load Fraud Detection Model

model = joblib.load(
    "../models/random_forest_fraud_detector.pkl"
)

print("Model Loaded Successfully")

Model Loaded Successfully


In [4]:
# Prepare Model Features

df_model = df.copy()

df_model = df_model.drop(
    columns=[
        "nameOrig",
        "nameDest"
    ]
)

categorical_columns = [
    "type",
    "customer_type",
    "merchant_type",
    "age_group",
    "customer_segment",
    "channel",
    "device_type",
    "risk_region"
]

for column in categorical_columns:

    encoder = LabelEncoder()

    df_model[column] = encoder.fit_transform(
        df_model[column].astype(str)
    )

print("Encoding Complete")

Encoding Complete


In [5]:
X = df_model.drop(
    columns=["isFraud"]
)

print(X.shape)

(6362620, 21)


In [6]:
# Generate Fraud Probability

fraud_probability = model.predict_proba(
    X
)[:, 1]

df["fraud_probability"] = fraud_probability

df[
    ["fraud_probability"]
].head()

,fraud_probability
0,0.000000
1,0.000000
2,0.999819
3,0.995939
4,0.009130


In [7]:
# Generate Risk Score

df["risk_score"] = (
    df["fraud_probability"] * 100
).round(2)

df[
    [
        "fraud_probability",
        "risk_score"
    ]
].head()

,fraud_probability,risk_score
0,0.000000,0.00
1,0.000000,0.00
2,0.999819,99.98
3,0.995939,99.59
4,0.009130,0.91


In [8]:
# Create Risk Categories

def risk_category(score):

    if score >= 90:
        return "Critical"

    elif score >= 70:
        return "High"

    elif score >= 40:
        return "Medium"

    return "Low"

df["risk_category"] = (
    df["risk_score"]
    .apply(risk_category)
)

df["risk_category"].value_counts()

risk_category
Low         6354408
Critical       7711
High            443
Medium           58
Name: count, dtype: int64

In [9]:
# Create Alert Priorities

def alert_priority(category):

    if category == "Critical":
        return "Immediate"

    elif category == "High":
        return "Urgent"

    elif category == "Medium":
        return "Review"

    return "Normal"

df["alert_priority"] = (
    df["risk_category"]
    .apply(alert_priority)
)

In [10]:
# Create Investigation Recommendations

def recommendation(row):

    score = row["risk_score"]

    if score >= 90:
        return (
            "Block Transaction Immediately and Escalate Investigation"
        )

    elif score >= 70:
        return (
            "Require Multi-Factor Verification"
        )

    elif score >= 40:
        return (
            "Manual Review Recommended"
        )

    return (
        "Approve Transaction"
    )

In [11]:
df["recommended_action"] = df.apply(
    recommendation,
    axis=1
)

In [12]:
# Generate Fraud Alerts

def alert_message(row):

    score = row["risk_score"]

    if score >= 90:

        return (
            f"CRITICAL FRAUD ALERT | Risk Score: {score}"
        )

    elif score >= 70:

        return (
            f"HIGH RISK TRANSACTION | Risk Score: {score}"
        )

    elif score >= 40:

        return (
            f"MEDIUM RISK TRANSACTION | Risk Score: {score}"
        )

    return (
        f"LOW RISK TRANSACTION | Risk Score: {score}"
    )

In [13]:
df["alert_message"] = df.apply(
    alert_message,
    axis=1
)

In [14]:
# Create Investigation Flags

df["review_required"] = np.where(
    df["risk_score"] >= 40,
    "Yes",
    "No"
)

df["investigation_required"] = np.where(
    df["risk_score"] >= 70,
    "Yes",
    "No"
)

In [15]:
# Create Fraud Severity Levels

def fraud_severity(score):

    if score >= 95:
        return "Extreme"

    elif score >= 80:
        return "Severe"

    elif score >= 60:
        return "Moderate"

    elif score >= 40:
        return "Minor"

    return "No Risk"

In [16]:
df["fraud_severity"] = (
    df["risk_score"]
    .apply(fraud_severity)
)

df["fraud_severity"].value_counts()

fraud_severity
No Risk     6354408
Extreme        6468
Severe         1565
Moderate        144
Minor            35
Name: count, dtype: int64

In [17]:
# Review Generated Risk Attributes

df[
    [
        "fraud_probability",
        "risk_score",
        "risk_category",
        "fraud_severity",
        "alert_priority",
        "recommended_action",
        "review_required",
        "investigation_required"
    ]
].head(10)

,fraud_probability,risk_score,risk_category,fraud_severity,alert_priority,recommended_action,review_required,investigation_required
0,0.000000,0.00,Low,No Risk,Normal,Approve Transaction,No,No
1,0.000000,0.00,Low,No Risk,Normal,Approve Transaction,No,No
2,0.999819,99.98,Critical,Extreme,Immediate,Block Transaction Immediately and Escalate Inv...,Yes,Yes
3,0.995939,99.59,Critical,Extreme,Immediate,Block Transaction Immediately and Escalate Inv...,Yes,Yes
4,0.009130,0.91,Low,No Risk,Normal,Approve Transaction,No,No
5,0.000000,0.00,Low,No Risk,Normal,Approve Transaction,No,No
6,0.000000,0.00,Low,No Risk,Normal,Approve Transaction,No,No
7,0.000000,0.00,Low,No Risk,Normal,Approve Transaction,No,No
8,0.000000,0.00,Low,No Risk,Normal,Approve Transaction,No,No
9,0.008952,0.90,Low,No Risk,Normal,Approve Transaction,No,No


In [18]:
# Export Risk Scored Dataset

df.to_csv(
    "../data/processed/fraud_risk_scored.csv",
    index=False
)

print("Risk Dataset Saved Successfully")

Risk Dataset Saved Successfully


In [ ]:
print(df.shape)
print(df.columns.tolist(
    
))

(6362620, 33)
['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud', 'transaction_hour', 'transaction_day', 'transaction_week', 'balance_error_org', 'balance_error_dest', 'customer_type', 'merchant_type', 'age_group', 'customer_segment', 'channel', 'device_type', 'risk_region', 'account_tenure_months', 'fraud_probability', 'risk_score', 'risk_category', 'alert_priority', 'recommended_action', 'alert_message', 'review_required', 'investigation_required', 'fraud_severity']


## Conclusion

Fraud probabilities generated by the machine learning model were converted into business-friendly risk scores and investigation recommendations.

The final dataset contains fraud risk indicators, alert priorities, fraud severity levels, and operational recommendations that can support real-time monitoring, fraud investigations, executive reporting, and Power BI dashboard development.

The risk-scored dataset will serve as the primary source for fraud analytics and business intelligence reporting.